In [0]:
# STEP 8: CREATE UNIFIED GOLD TABLE
print("\n" + "-"*70)
print("STEP 5: CREATING UNIFIED GOLD TABLE")
print("-"*70)

print("\nJoining all 6 gold tables into a single comprehensive table...")

# Import required functions (if not already imported at the top)
from pyspark.sql.functions import col, when, round, avg, max, min, count, sum as spark_sum, desc, first

try:
    # Read all gold tables
    customer_agg_df = spark.table("credit_risk.gold_customer_agg")
    risk_segments_df = spark.table("credit_risk.gold_risk_segments")
    education_agg_df = spark.table("credit_risk.gold_education_agg")
    age_group_agg_df = spark.table("credit_risk.gold_age_group_agg")
    gender_agg_df = spark.table("credit_risk.gold_gender_agg")
    scorecard_df = spark.table("credit_risk.gold_scorecard")
    
    # Create derived columns for joining
    customer_with_segments = customer_agg_df \
        .withColumn("spending_segment",
            when(col("avg_bill_6months") > 50000, "High Spender")
            .when(col("avg_bill_6months") > 20000, "Medium Spender")
            .otherwise("Low Spender")
        ) \
        .withColumn("payment_status_segment",
            when(col("latest_pay_status").isin(2, 0, -1, -2), "Good Standing")
            .when(col("latest_pay_status") == 1, "Revolving")
            .otherwise("Delinquent")
        ) \
        .withColumn("age_group",
            when(col("AGE") < 25, "18-24")
            .when(col("AGE") < 35, "25-34")
            .when(col("AGE") < 45, "35-44")
            .when(col("AGE") < 55, "45-54")
            .otherwise("55+")
        ) \
        .withColumn("credit_utilization",
            when(col("utilization_ratio") > 80, "High Utilization")
            .when(col("utilization_ratio") > 50, "Medium Utilization")
            .otherwise("Low Utilization")
        ) \
        .withColumn("latest_payment_status",
            when(col("latest_pay_status") == 0, "On-Time")
            .when(col("latest_pay_status") == 1, "1 Month Late")
            .when(col("latest_pay_status") == 2, "2+ Months Late")
            .otherwise("Revolving")
        ) \
        .withColumnRenamed("SEX", "gender")
    
    # Join 1: Add risk segment statistics
    unified_df = customer_with_segments.join(
        risk_segments_df.select(
            col("spending_segment"),
            col("payment_status_segment"),
            col("default_flag"),
            col("customer_count").alias("risk_segment_customer_count"),
            col("avg_credit_limit").alias("risk_segment_avg_credit_limit"),
            col("avg_bill_amount").alias("risk_segment_avg_bill_amount"),
            col("avg_age").alias("risk_segment_avg_age"),
            col("default_rate_pct").alias("risk_segment_default_rate_pct")
        ),
        on=["spending_segment", "payment_status_segment", "default_flag"],
        how="left"
    )
    
    # Join 2: Add education statistics
    unified_df = unified_df.join(
        education_agg_df.select(
            col("EDUCATION"),
            col("customer_count").alias("education_customer_count"),
            col("avg_credit_limit").alias("education_avg_credit_limit"),
            col("avg_bill_amount").alias("education_avg_bill_amount"),
            col("avg_age").alias("education_avg_age"),
            col("default_rate_pct").alias("education_default_rate_pct")
        ),
        on="EDUCATION",
        how="left"
    )
    
    # Join 3: Add age group statistics
    # Create alias for age group table and rename default_flag before join
    age_group_prepared = age_group_agg_df.select(
        col("age_group").alias("age_group_join"),
        col("default_flag").alias("age_default_flag_join"),
        col("customer_count").alias("age_group_customer_count"),
        col("avg_credit_limit").alias("age_group_avg_credit_limit"),
        col("avg_bill_amount").alias("age_group_avg_bill_amount"),
        col("avg_age").alias("age_group_avg_age")
    )
    
    unified_df = unified_df.join(
        age_group_prepared,
        on=(unified_df.age_group == age_group_prepared.age_group_join) & 
           (unified_df.default_flag == age_group_prepared.age_default_flag_join),
        how="left"
    ).drop("age_group_join", "age_default_flag_join")
    
    # Join 4: Add gender statistics
    # Create alias for gender table and rename default_flag before join
    gender_prepared = gender_agg_df.select(
        col("gender").alias("gender_join"),
        col("default_flag").alias("gender_default_flag_join"),
        col("customer_count").alias("gender_customer_count"),
        col("avg_credit_limit").alias("gender_avg_credit_limit"),
        col("avg_bill_amount").alias("gender_avg_bill_amount"),
        col("avg_age").alias("gender_avg_age")
    )
    
    unified_df = unified_df.join(
        gender_prepared,
        on=(unified_df.gender == gender_prepared.gender_join) & 
           (unified_df.default_flag == gender_prepared.gender_default_flag_join),
        how="left"
    ).drop("gender_join", "gender_default_flag_join")
    
    # Join 5: Add scorecard statistics
    # Create alias for scorecard table and rename default_flag before join
    scorecard_prepared = scorecard_df.select(
        col("credit_utilization").alias("credit_utilization_join"),
        col("latest_payment_status").alias("latest_payment_status_join"),
        col("default_flag").alias("scorecard_default_flag_join"),
        col("count").alias("scorecard_count"),
        col("avg_age").alias("scorecard_avg_age"),
        col("avg_limit").alias("scorecard_avg_limit"),
        col("avg_bill").alias("scorecard_avg_bill")
    )
    
    unified_df = unified_df.join(
        scorecard_prepared,
        on=(unified_df.credit_utilization == scorecard_prepared.credit_utilization_join) & 
           (unified_df.latest_payment_status == scorecard_prepared.latest_payment_status_join) & 
           (unified_df.default_flag == scorecard_prepared.scorecard_default_flag_join),
        how="left"
    ).drop("credit_utilization_join", "latest_payment_status_join", "scorecard_default_flag_join")
    
    # Save unified table
    print("\nWriting unified table to catalog...")
    unified_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("credit_risk.gold_unified")
    
    # Add table comment
    spark.sql("""
        ALTER TABLE credit_risk.gold_unified
        SET TBLPROPERTIES (
            'description' = 'Unified gold table with all customer, risk, demographic, and scorecard data'
        )
    """)
    
    record_count = unified_df.count()
    column_count = len(unified_df.columns)
    print(f"  Success: {record_count:,} records")
    print(f"  Columns: {column_count}")
    
    # Display sample
    print("\n" + "-"*70)
    print("UNIFIED GOLD TABLE - SAMPLE DATA")
    print("-"*70)
    display(unified_df.limit(10))
    
    # Display schema
    print("\n" + "-"*70)
    print("UNIFIED GOLD TABLE - SCHEMA")
    print("-"*70)
    unified_df.printSchema()
    
    print("\n✓ Unified table created: credit_risk.gold_unified")
    
except Exception as e:
    print(f"Error creating unified table: {e}")
    import traceback
    traceback.print_exc()

# Update final summary
print("\n" + "="*70)
print("GOLD LAYER PIPELINE COMPLETE!")
print("="*70)

print("\n✓ TABLES NOW IN CATALOG:")
print("  1. credit_risk.gold_customer_agg")
print("  2. credit_risk.gold_risk_segments")
print("  3. credit_risk.gold_education_agg")
print("  4. credit_risk.gold_age_group_agg")
print("  5. credit_risk.gold_gender_agg")
print("  6. credit_risk.gold_scorecard")
print("  7. credit_risk.gold_unified (NEW - All tables combined)")

print("\n✓ UNIFIED TABLE CONTAINS:")
print("  - All customer-level metrics")
print("  - Risk segment statistics")
print("  - Education demographic stats")
print("  - Age group statistics")
print("  - Gender statistics")
print("  - Scorecard metrics")
try:
    print(f"  - Total columns: {column_count}")
except:
    pass

print("\n" + "="*70)


----------------------------------------------------------------------
STEP 5: CREATING UNIFIED GOLD TABLE
----------------------------------------------------------------------

Joining all 6 gold tables into a single comprehensive table...

Writing unified table to catalog...
  Success: 30,000 records
  Columns: 45

----------------------------------------------------------------------
UNIFIED GOLD TABLE - SAMPLE DATA
----------------------------------------------------------------------


EDUCATION,spending_segment,payment_status_segment,default_flag,ID,LIMIT_BAL,gender,MARRIAGE,AGE,latest_pay_status,avg_pay_status,avg_monthly_bill,max_monthly_bill,min_monthly_bill,avg_bill_6months,avg_monthly_payment,max_monthly_payment,min_monthly_payment,utilization_ratio,record_count,age_group,credit_utilization,latest_payment_status,risk_segment_customer_count,risk_segment_avg_credit_limit,risk_segment_avg_bill_amount,risk_segment_avg_age,risk_segment_default_rate_pct,education_customer_count,education_avg_credit_limit,education_avg_bill_amount,education_avg_age,education_default_rate_pct,age_group_customer_count,age_group_avg_credit_limit,age_group_avg_bill_amount,age_group_avg_age,gender_customer_count,gender_avg_credit_limit,gender_avg_bill_amount,gender_avg_age,scorecard_count,scorecard_avg_age,scorecard_avg_limit,scorecard_avg_bill
4,High Spender,Good Standing,N,114.0,100000.0,F,Single,24.0,0.0,0.0,52128.0,52128.0,52128.0,55864.333333333336,2000.0,2000.0,2000.0,52.13,3,18-24,Medium Utilization,On-Time,18726,223373.92,122196.27,35.9,0.0,89958,167461.14,44992.86,35.5,22.13,5865,64143.22,25966.52,23.2,43047,179726.53,43999.21,34.8,11304,35.4,130581.0,85183.0
4,Medium Spender,Good Standing,Y,121.0,50000.0,M,Single,37.0,2.0,2.0,46004.0,46004.0,46004.0,48374.166666666664,1000.0,1000.0,1000.0,92.01,3,35-44,High Utilization,2+ Months Late,3786,78851.03,33403.78,34.5,100.0,89958,167461.14,44992.86,35.5,22.13,5913,153910.54,50286.82,39.1,8619,125895.47,44988.59,36.8,1929,35.1,98040.0,90098.0
4,High Spender,Good Standing,N,205.0,360000.0,F,Married,48.0,0.0,0.0,226430.0,226430.0,226430.0,200463.16666666666,9100.0,9100.0,9100.0,62.9,3,45-54,Medium Utilization,On-Time,18726,223373.92,122196.27,35.9,0.0,89958,167461.14,44992.86,35.5,22.13,9660,182665.22,47222.6,48.7,43047,179726.53,43999.21,34.8,11304,35.4,130581.0,85183.0
4,High Spender,Good Standing,Y,239.0,240000.0,F,Married,50.0,0.0,0.0,234205.0,234205.0,234205.0,218871.16666666666,10116.0,10116.0,10116.0,97.59,3,45-54,High Utilization,On-Time,4233,181444.14,126148.53,36.5,100.0,89958,167461.14,44992.86,35.5,22.13,3039,132152.02,42163.87,48.8,11289,133327.13,42311.45,34.9,1689,35.2,101470.0,93335.0
4,High Spender,Good Standing,N,595.0,110000.0,M,Single,46.0,0.0,0.0,56700.0,56700.0,56700.0,59432.0,2681.0,2681.0,2681.0,51.55,3,45-54,Medium Utilization,On-Time,18726,223373.92,122196.27,35.9,0.0,89958,167461.14,44992.86,35.5,22.13,9660,182665.22,47222.6,48.7,27045,175510.37,47642.09,36.4,11304,35.4,130581.0,85183.0
4,Low Spender,Revolving,N,1035.0,60000.0,F,Other,39.0,1.0,1.0,-1540.0,-1540.0,-1540.0,-1168.3333333333333,0.0,0.0,0.0,-2.57,3,35-44,Low Utilization,1 Month Late,5097,169140.67,4405.53,35.2,0.0,89958,167461.14,44992.86,35.5,22.13,21141,204660.14,47785.89,39.0,43047,179726.53,43999.21,34.8,5370,35.4,188123.0,9866.0
4,Low Spender,Good Standing,N,1451.0,40000.0,F,Married,25.0,0.0,0.0,11273.0,11273.0,11273.0,13066.833333333334,1520.0,1520.0,1520.0,28.18,3,25-34,Low Utilization,On-Time,28809,176419.45,6994.56,35.4,0.0,89958,167461.14,44992.86,35.5,22.13,31110,180718.42,46418.3,29.4,43047,179726.53,43999.21,34.8,19266,34.6,197183.0,41561.0
4,Medium Spender,Good Standing,N,1487.0,230000.0,M,Single,32.0,0.0,0.0,44734.0,44734.0,44734.0,40991.833333333336,10120.0,10120.0,10120.0,19.45,3,25-34,Low Utilization,On-Time,14859,136157.08,33452.31,34.9,0.0,89958,167461.14,44992.86,35.5,22.13,31110,180718.42,46418.3,29.4,27045,175510.37,47642.09,36.4,19266,34.6,197183.0,41561.0
4,Medium Spender,Good Standing,N,1557.0,50000.0,M,Single,37.0,0.0,0.0,37774.0,37774.0,37774.0,33699.833333333336,1976.0,1976.0,1976.0,75.55,3,35-44,Medium Utilization,On-Time,14859,136157.08,33452.31,34.9,0.0,89958,167461.14,44992.86,35.5,22.13,21141,204660.14,47785.89,39.0,27045,175510.37,47642.09,36.4,11304,35.4,130581.0,85183.0
4,Low Spender,Good Standing,Y,1570.0,30000.0,F,Single,23.0,-1.0,-1.0,3226.0,3226.0,3226.0,9294.5,3596.0,3596.0,3596.0,10.75,3,18-24,Low Utilization,Revolving,7134,134108.49,6718.0,35.8,100.


----------------------------------------------------------------------
UNIFIED GOLD TABLE - SCHEMA
----------------------------------------------------------------------
root
 |-- EDUCATION: string (nullable = true)
 |-- spending_segment: string (nullable = false)
 |-- payment_status_segment: string (nullable = false)
 |-- default_flag: string (nullable = true)
 |-- ID: double (nullable = true)
 |-- LIMIT_BAL: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- MARRIAGE: string (nullable = true)
 |-- AGE: double (nullable = true)
 |-- latest_pay_status: double (nullable = true)
 |-- avg_pay_status: double (nullable = true)
 |-- avg_monthly_bill: double (nullable = true)
 |-- max_monthly_bill: double (nullable = true)
 |-- min_monthly_bill: double (nullable = true)
 |-- avg_bill_6months: double (nullable = true)
 |-- avg_monthly_payment: double (nullable = true)
 |-- max_monthly_payment: double (nullable = true)
 |-- min_monthly_payment: double (nullable = true)
 |-- ut